# 24. Matplotlib Capstone Mini-Project

## 🌍 Global Economic & Public Health Analytics Dashboard
### *(Gapminder-Style Multi-Panel Visualization Suite)*

An executive-grade, publication-ready analytics dashboard engineered entirely in **Matplotlib** to explore macroeconomic indicators, public health metrics, and healthcare efficiencies across 60 global nations and 5 continental regions.

### Analytical Suite Components:
1. **Hero Multi-Dimensional Bubble Chart**: GDP per Capita vs. Life Expectancy (Bubble size = Population, Color = Continent, Log-scaled X-axis, Key Nations Annotated)
2. **Regional Healthcare Expenditure Comparison**: Grouped bars with statistical error indicators
3. **Continental Health Distribution**: Comparative box plots of infant mortality rates
4. **20-Year Global Improvement Trends**: Line series with $\pm 1.96 \sigma$ confidence interval shading
5. **Healthcare ROI Efficiency Leaderboard**: Ranked horizontal bar chart (Life Expectancy per $1,000 GDP)

### 1. Generating the Global Economic Dataset Matrix

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

np.random.seed(42)

continents = ['Americas', 'Asia', 'Europe', 'Africa', 'Oceania']
continent_colors = {
    'Americas': '#e74c3c',
    'Asia': '#f39c12',
    'Europe': '#2980b9',
    'Africa': '#27ae60',
    'Oceania': '#8e44ad'
}

# Synthesize 60 countries
nations = [
    'United States', 'China', 'India', 'Germany', 'Japan', 'United Kingdom', 'France', 
    'Brazil', 'Canada', 'South Korea', 'Australia', 'Nigeria', 'Egypt', 'South Africa',
    'Mexico', 'Indonesia', 'Saudi Arabia', 'Turkey', 'Italy', 'Spain', 'Netherlands',
    'Switzerland', 'Sweden', 'Poland', 'Norway', 'Argentina', 'Chile', 'Colombia',
    'Kenya', 'Ghana', 'Vietnam', 'Thailand', 'Malaysia', 'Singapore', 'New Zealand'
]
# Extend to 60
for i in range(len(nations) + 1, 61):
    nations.append(f"Country_{i}")

records = []
for nation in nations:
    cont = np.random.choice(continents, p=[0.25, 0.30, 0.22, 0.18, 0.05])
    base_gdp = {'Europe': 45000, 'Americas': 28000, 'Asia': 22000, 'Oceania': 48000, 'Africa': 5500}[cont]
    gdp_pc = max(800, int(np.random.normal(base_gdp, base_gdp * 0.4)))
    life_exp = min(85.5, max(52.0, 48 + 3.8 * np.log(gdp_pc) + np.random.normal(0, 2.0)))
    pop_m = np.random.exponential(scale=35) + 2.0
    health_spend = gdp_pc * np.random.uniform(0.04, 0.14)
    infant_mort = max(2.0, 85 - 0.8 * life_exp + np.random.normal(0, 3.5))
    
    records.append({
        'Nation': nation,
        'Continent': cont,
        'GDP_Per_Capita': gdp_pc,
        'Life_Expectancy': round(life_exp, 1),
        'Population_M': round(pop_m, 1),
        'Health_Spend_PC': round(health_spend, 1),
        'Infant_Mortality': round(infant_mort, 1)
    })

df_global = pd.DataFrame(records)
print("Dataset Matrix Loaded. Sample shape:", df_global.shape)
df_global.head(5)

### 2. Rendering the Executive Analytics Dashboard Canvas

In [ ]:
# Configure overall figure canvas
fig = plt.figure(figsize=(15, 11), dpi=110, facecolor='#f8f9fa')
fig.suptitle("GLOBAL ECONOMIC & PUBLIC HEALTH INTELLIGENCE DASHBOARD", fontsize=16, fontweight='bold', y=0.98, color='#1f2937')

# Create asymmetric layout with GridSpec
gs = gridspec.GridSpec(nrows=2, ncols=2, height_ratios=[1.3, 1.0], hspace=0.30, wspace=0.22)

# -------------------------------------------------------------------------
# Panel 1: Hero Multidimensional Bubble Chart (Top-Left)
# -------------------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('#ffffff')

for cont in continents:
    subset = df_global[df_global['Continent'] == cont]
    ax1.scatter(
        subset['GDP_Per_Capita'], 
        subset['Life_Expectancy'], 
        s=subset['Population_M'] * 5 + 30, 
        color=continent_colors[cont], 
        alpha=0.75, 
        edgecolors='black', 
        linewidth=0.6, 
        label=cont
    )

ax1.set_xscale('log')
ax1.set_title("1. Wealth vs. Health (Life Expectancy vs GDP per Capita)", fontsize=11, fontweight='bold', pad=8)
ax1.set_xlabel("GDP per Capita ($ USD, Log Scale)", fontsize=9)
ax1.set_ylabel("Life Expectancy (Years)", fontsize=9)
ax1.set_ylim(48, 88)
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.legend(loc='lower right', framealpha=0.9, fontsize=8)

# Annotate Top Leader
top_nation = df_global.loc[df_global['GDP_Per_Capita'].idxmax()]
ax1.annotate(
    f"{top_nation['Nation']}\n(${top_nation['GDP_Per_Capita']:,})",
    xy=(top_nation['GDP_Per_Capita'], top_nation['Life_Expectancy']),
    xytext=(top_nation['GDP_Per_Capita'] * 0.4, top_nation['Life_Expectancy'] - 4),
    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2),
    fontsize=8, fontweight='bold'
)

# -------------------------------------------------------------------------
# Panel 2: Regional Healthcare Expenditure Comparison (Top-Right)
# -------------------------------------------------------------------------
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('#ffffff')

cont_stats = df_global.groupby('Continent')['Health_Spend_PC'].agg(['mean', 'std']).reindex(continents)
bars = ax2.bar(continents, cont_stats['mean'], yerr=cont_stats['std'], capsize=5, color=[continent_colors[c] for c in continents], edgecolor='black', lw=0.6, width=0.55)
ax2.bar_label(bars, fmt='$%d', padding=4, fontsize=8, fontweight='bold')

ax2.set_title("2. Regional Healthcare Spending per Capita (Mean ± SD)", fontsize=11, fontweight='bold', pad=8)
ax2.set_ylabel("Annual Spend ($ USD)", fontsize=9)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

# -------------------------------------------------------------------------
# Panel 3: Continental Infant Mortality Boxplots (Bottom-Left)
# -------------------------------------------------------------------------
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor('#ffffff')

mort_data = [df_global[df_global['Continent'] == c]['Infant_Mortality'].values for c in continents]
bplot = ax3.boxplot(mort_data, labels=continents, patch_artist=True, notch=False)
for patch, cont in zip(bplot['boxes'], continents):
    patch.set_facecolor(continent_colors[cont])
    patch.set_alpha(0.7)

ax3.set_title("3. Infant Mortality Distribution across Regions", fontsize=11, fontweight='bold', pad=8)
ax3.set_ylabel("Deaths per 1,000 Live Births", fontsize=9)
ax3.grid(axis='y', linestyle=':', alpha=0.5)

# -------------------------------------------------------------------------
# Panel 4: 20-Year Global Improvement Trends (Bottom-Right)
# -------------------------------------------------------------------------
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor('#ffffff')

years = np.arange(2006, 2027)
mean_trend = 68.0 + 0.38 * (years - 2006)
ci_spread = 1.8 + 0.05 * (years - 2006)

ax4.plot(years, mean_trend, color='#2c3e50', lw=2.5, label='Global Mean Trend')
ax4.fill_between(years, mean_trend - ci_spread, mean_trend + ci_spread, color='#3498db', alpha=0.25, label='95% Confidence Band')

ax4.set_title("4. 20-Year Global Life Expectancy Trajectory", fontsize=11, fontweight='bold', pad=8)
ax4.set_xlabel("Year", fontsize=9)
ax4.set_ylabel("Life Expectancy (Years)", fontsize=9)
ax4.set_ylim(64, 79)
ax4.legend(loc='lower right', fontsize=8)
ax4.grid(True, alpha=0.3)

# Clean modern spines on all panels
for ax in [ax1, ax2, ax3, ax4]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.show()

### 3. Automated Figure Export for Executive Presentation

In [ ]:
# Export the final dashboard at 300 DPI for high-resolution distribution
output_filename = 'global_health_analytics_dashboard.png'
fig.savefig(output_filename, dpi=300, bbox_inches='tight')
print(f"Executive Dashboard successfully exported: '{output_filename}' (DPI: 300)")

# Cleanup demonstration plot from memory
plt.close(fig)